In [7]:
import pandas as pd
# 1. Load the dataset.
df = pd.read_csv('data/iot_sensor_data_raw.csv')
# 2. Display its dimensions and structure.
print('Shape (rows, columns):', df.shape)
df.info()
# 3. Convert Timestamp into Pandas datetime.
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
print(df['Timestamp'].dtype)
print(df['Timestamp'].min(), 'to', df['Timestamp'].max())

# 4. Check whether the timestamps are correctly ordered.
is_sorted = df['Timestamp'].is_monotonic_increasing
print('Are timestamps in increasing order?', is_sorted)

diffs = df['Timestamp'].diff()
out_of_order = (diffs < pd.Timedelta(0)).sum()
print('Number of out-of-order transitions:', out_of_order)

# 5. Identify missing sensor readings.
missing = df.isnull().sum()
print(missing[missing > 0])

# 6. Calculate the percentage of missing values for each sensor. 
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].round(2))

Shape (rows, columns): (20000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Timestamp       20000 non-null  str    
 1   Device_ID       20000 non-null  str    
 2   Temperature     19701 non-null  float64
 3   Humidity        19700 non-null  float64
 4   Pressure        19700 non-null  float64
 5   Vibration       19702 non-null  float64
 6   Battery_Level   20000 non-null  float64
 7   Location        20000 non-null  str    
 8   Machine_Status  20000 non-null  str    
dtypes: float64(5), str(4)
memory usage: 1.4 MB
datetime64[us]
2026-01-01 00:00:00 to 2026-07-28 07:45:00
Are timestamps in increasing order? False
Number of out-of-order transitions: 10063
Temperature    299
Humidity       300
Pressure       300
Vibration      298
dtype: int64
Temperature    1.50
Humidity       1.50
Pressure       1.50
Vibration      1.49
dtype: float64


In [ ]:
# 7. Handle missing sensor values using an appropriate method.
df = df.sort_values(['Device_ID', 'Timestamp']).reset_index(drop=True)

sensor_cols = ['Temperature', 'Humidity', 'Pressure', 'Vibration']

for col in sensor_cols:
    df[col] = df.groupby('Device_ID')[col].transform(lambda x: x.interpolate(method='linear'))

for col in sensor_cols:
    df[col] = df[col].fillna(df[col].median())

print(df[sensor_cols].isnull().sum())

# 8. Identify abnormal temperature readings.
Q1 = df['Temperature'].quantile(0.25)
Q3 = df['Temperature'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

abnormal_temp = df[(df['Temperature'] < lower) | (df['Temperature'] > upper)]
print('Abnormal temperature readings:', len(abnormal_temp))

# 9. Identify abnormal vibration readings.
Q1 = df['Vibration'].quantile(0.25)
Q3 = df['Vibration'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

abnormal_vib = df[(df['Vibration'] < lower) | (df['Vibration'] > upper)]
print('Abnormal vibration readings:', len(abnormal_vib))

# 10. Identify machines with critically low battery levels.
critical_battery = df[df['Battery_Level'] < 20]
print('Critical battery readings:', len(critical_battery))

critical_by_device = critical_battery.groupby('Device_ID').size().sort_values(ascending=False)
print(critical_by_device)

# 11. Explain how you decided what constitutes an abnormal reading.
I used the IQR method (1.5×IQR rule) for Temperature and Vibration, since it's a data-driven statistical approach that adapts to each sensor's own distribution rather than relying on a guessed fixed threshold. For Vibration, I also flagged the negative value (-0.48) separately as a domain-knowledge issue, since vibration magnitude can't physically be negative, that's a sensor error, not a statistical outlier. For Battery_Level, I used the assignment's own defined scale (<20 = Critical) instead of IQR, since battery percentage already has a meaningful, bounded 0–100 scale where a business rule makes more sense than a statistical one.

In [17]:
# 12. Calculate the average temperature by hour.
df['Hour'] = df['Timestamp'].dt.hour
avg_temp_by_hour = df.groupby('Hour')['Temperature'].mean()
print(avg_temp_by_hour.round(2))

# 13. Calculate the average temperature for each device.
avg_temp_by_device = df.groupby('Device_ID')['Temperature'].mean().sort_values(ascending=False)
print(avg_temp_by_device.round(2))

# 14. Calculate the average vibration for each device.
avg_vib_by_device = df.groupby('Device_ID')['Vibration'].mean().sort_values(ascending=False)
print(avg_vib_by_device.round(3))

# 15. Find the maximum temperature recorded by each device.
max_temp_by_device = df.groupby('Device_ID')['Temperature'].max().sort_values(ascending=False)
print(max_temp_by_device.round(2))

# 16. Find the minimum battery level for each device.
min_battery_by_device = df.groupby('Device_ID')['Battery_Level'].min().sort_values()
print(min_battery_by_device.round(2))

# 17. Determine which device has the highest average vibration.
avg_vib_by_device = df.groupby('Device_ID')['Vibration'].mean()
print('Highest average vibration device:', avg_vib_by_device.idxmax())
print('Value:', round(avg_vib_by_device.max(), 3))
# 18. Determine which factory has the highest average temperature.
avg_temp_by_factory = df.groupby('Location')['Temperature'].mean().sort_values(ascending=False)
print('Highest average temperature factory:', avg_temp_by_factory.idxmax())

Hour
0     70.10
1     70.53
2     70.01
3     69.71
4     69.83
5     70.28
6     70.26
7     70.67
8     70.45
9     70.97
10    69.93
11    69.89
12    70.02
13    70.13
14    69.82
15    70.23
16    69.96
17    70.57
18    70.20
19    69.96
20    70.15
21    70.12
22    71.15
23    70.40
Name: Temperature, dtype: float64
Device_ID
DEV_004    70.59
DEV_002    70.38
DEV_008    70.32
DEV_001    70.26
DEV_005    70.14
DEV_007    70.12
DEV_006    70.01
DEV_003    69.94
Name: Temperature, dtype: float64
Device_ID
DEV_002    2.568
DEV_006    2.557
DEV_001    2.554
DEV_003    2.540
DEV_008    2.528
DEV_004    2.527
DEV_007    2.523
DEV_005    2.495
Name: Vibration, dtype: float64
Device_ID
DEV_001    159.75
DEV_003    157.60
DEV_007    153.88
DEV_008    153.62
DEV_005    152.97
DEV_004    151.97
DEV_002    151.82
DEV_006    136.00
Name: Temperature, dtype: float64
Device_ID
DEV_008    2.01
DEV_007    2.15
DEV_002    2.15
DEV_006    2.20
DEV_001    2.26
DEV_005    2.96
DEV_003    3.67
DEV_0

In [20]:
# 19. Create a  column:
# Battery Status
# >= 50 Healthy
# 20–49 Moderate
# <20 Critical
def battery_status(r):
    if r >= 50:
        return 'Healthy'
    elif r >= 20:
        return 'Moderate'
    else:
        return 'Critical'

df['Battery_Status'] = df['Battery_Level'].apply(battery_status)
print(df['Battery_Status'].value_counts())

# 20. Create a Temperature_Status column.
Q1 = df['Temperature'].quantile(0.25)
Q3 = df['Temperature'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR          # Critical bounds
mild_lower, mild_upper = Q1 - 1.0*IQR, Q3 + 1.0*IQR  # Warning bounds

def temp_status(t):
    if t < lower or t > upper:
        return 'Critical'
    elif t < mild_lower or t > mild_upper:
        return 'Warning'
    else:
        return 'Normal'

df['Temperature_Status'] = df['Temperature'].apply(temp_status)
print(df['Temperature_Status'].value_counts())
# 21. Create a Vibration_Status column.
Q1 = df['Vibration'].quantile(0.25)
Q3 = df['Vibration'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
mild_lower, mild_upper = Q1 - 1.0*IQR, Q3 + 1.0*IQR

def vib_status(v):
    if v < lower or v > upper:
        return 'Critical'
    elif v < mild_lower or v > mild_upper:
        return 'Warning'
    else:
        return 'Normal'

df['Vibration_Status'] = df['Vibration'].apply(vib_status)
print(df['Vibration_Status'].value_counts())
# 22. Create an overall Machine_Health column:
# Normal
# Warning
# Critical
# based on appropriate sensor conditions.

battery_map = {'Healthy': 'Normal', 'Moderate': 'Warning', 'Critical': 'Critical'}
df['Battery_Status_Mapped'] = df['Battery_Status'].map(battery_map)

def machine_health(row):
    statuses = [row['Battery_Status_Mapped'], row['Temperature_Status'], row['Vibration_Status']]
    if 'Critical' in statuses:
        return 'Critical'
    elif 'Warning' in statuses:
        return 'Warning'
    else:
        return 'Normal'

df['Machine_Health'] = df.apply(machine_health, axis=1)
print(df['Machine_Health'].value_counts())

Battery_Status
Healthy     10978
Moderate     6705
Critical     2317
Name: count, dtype: int64
Temperature_Status
Normal      19031
Warning       773
Critical      196
Name: count, dtype: int64
Vibration_Status
Normal      19112
Warning       719
Critical      169
Name: count, dtype: int64
Machine_Health
Normal      9998
Warning     7373
Critical    2629
Name: count, dtype: int64


In [23]:
# 23. Find devices that have experienced critical conditions more than five times.
critical_counts = df[df['Machine_Health'] == 'Critical'].groupby('Device_ID').size().sort_values(ascending=False)
devices_over_5 = critical_counts[critical_counts > 5]
print(devices_over_5)

# 24. Find the factory with the highest number of abnormal sensor readings.
Q1t, Q3t = df['Temperature'].quantile(0.25), df['Temperature'].quantile(0.75)
IQRt = Q3t - Q1t
t_low, t_up = Q1t - 1.5*IQRt, Q3t + 1.5*IQRt

Q1v, Q3v = df['Vibration'].quantile(0.25), df['Vibration'].quantile(0.75)
IQRv = Q3v - Q1v
v_low, v_up = Q1v - 1.5*IQRv, Q3v + 1.5*IQRv
df['Abnormal_Sensor'] = (
    (df['Temperature'] < t_low) | (df['Temperature'] > t_up) |
    (df['Vibration'] < v_low) | (df['Vibration'] > v_up)
)

abnormal_by_factory = df[df['Abnormal_Sensor']].groupby('Location').size().sort_values(ascending=False)
print('Factory with highest abnormal sensor readings:', abnormal_by_factory.idxmax())

# 25. Calculate the percentage of time each device operates under warning/critical conditions.
pct_warning_critical = (
    df.groupby('Device_ID')['Machine_Health']
      .apply(lambda x: (x.isin(['Warning', 'Critical'])).mean() * 100)
      .sort_values(ascending=False)
)
print(pct_warning_critical.round(2))

# 26. Identify the device that requires the highest maintenance priority
df['Temp_Critical'] = (df['Temperature'] < t_low) | (df['Temperature'] > t_up)
df['Vib_Critical'] = (df['Vibration'] < v_low) | (df['Vibration'] > v_up)

maintenance_summary = df.groupby('Device_ID').agg(
    Temp_Critical_Count=('Temp_Critical', 'sum'),
    Vib_Critical_Count=('Vib_Critical', 'sum'),
    Max_Temp=('Temperature', 'max'),
    Max_Vibration=('Vibration', 'max')
)
maintenance_summary['Total_Abnormal_Events'] = (
    maintenance_summary['Temp_Critical_Count'] + maintenance_summary['Vib_Critical_Count']
)
maintenance_summary = maintenance_summary.sort_values('Total_Abnormal_Events', ascending=False)
print(maintenance_summary)

Device_ID
DEV_004    358
DEV_005    350
DEV_002    339
DEV_001    336
DEV_006    334
DEV_008    318
DEV_003    306
DEV_007    288
dtype: int64
Factory with highest abnormal sensor readings: Factory_A
Device_ID
DEV_006    51.69
DEV_002    50.20
DEV_003    50.17
DEV_004    50.16
DEV_005    49.90
DEV_008    49.38
DEV_007    49.34
DEV_001    49.29
Name: Machine_Health, dtype: float64
           Temp_Critical_Count  Vib_Critical_Count    Max_Temp  Max_Vibration  \
Device_ID                                                                       
DEV_001                     28                  28  159.746758      19.739138   
DEV_004                     27                  28  151.965273      16.349788   
DEV_005                     28                  17  152.974450      15.227651   
DEV_002                     16                  28  151.818550      18.470897   
DEV_006                     18                  26  136.003203      17.874138   
DEV_003                     29                  14